# Train an MLX Semantic-Segmentation Model

This notebook runs in a local Jupyter environment or Google Colab. It downloads a ZIP archive containing one paired image/mask dataset, validates the extracted layout, and trains through MLX's semantic-segmentation command API.

The archive may contain the dataset at its root or inside one wrapper directory. It must contain `train/images`, `train/masks`, `val/images`, and `val/masks`; `test/images` and `test/masks` are optional. Workspace and checkpoint paths include the selected `MODEL`, so each model has an isolated run. Run the cells from top to bottom.

## 1. Configuration

Set `DATASET_ZIP_URL` to a directly downloadable HTTP(S) ZIP file. `LOCAL_WORKSPACE_ROOT` and `GOOGLE_DRIVE_CHECKPOINT_ROOT` are parent directories: the notebook appends `MODEL` automatically.

In [ ]:
# Dataset download
DATASET_ZIP_URL = ""  # Example: https://example.com/my-segmentation-dataset.zip

# MLX source and model
REPO_URL = "https://github.com/ralampay/mlx.git"
MODEL = "unet"  # Run `python -m mlx --mode segmentation --action ls-models` for choices
INSTALL_MISSING_DEPENDENCIES = True

# Training
EPOCHS = 50
BATCH_SIZE = 4
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256
DEVICE = "auto"  # "auto", "cpu", "cuda", or "cuda:0"
LEARNING_RATE = 1e-3
PRETRAINED = False
COLORED = True
NUM_CLASSES = 2
CLASS_NAMES = "background,foreground"
RANDOM_SEED = 42

# Checkpoints
CHECKPOINT_STORAGE = "local"  # "local" or "google_drive"
RESUME_LATEST = True
LOCAL_WORKSPACE_ROOT = ""  # Empty uses <repo>/tmp/notebooks/train_semantic_segementation
GOOGLE_DRIVE_CHECKPOINT_ROOT = "/content/drive/MyDrive/mlx/semantic_segmentation/checkpoints"

## 2. Prepare the runtime

A local notebook reuses the surrounding MLX checkout. A standalone Colab notebook clones MLX when necessary. Missing Python dependencies are installed into the active notebook runtime.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path


def is_mlx_checkout(path: Path) -> bool:
    return (
        (path / "pyproject.toml").is_file()
        and (path / "mlx" / "modes" / "segmentation").is_dir()
    )


def find_mlx_checkout() -> Path | None:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents, current / "mlx"):
        if is_mlx_checkout(candidate):
            return candidate
    return None


try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True


def import_pytorch_runtime():
    try:
        import torch
        import torchvision
    except (AttributeError, ImportError, OSError, RuntimeError) as exc:
        recovery = (
            " In Google Colab, select Runtime > Disconnect and delete runtime, "
            "reconnect, and run the notebook from the first cell."
            if IN_COLAB
            else " Restart the Jupyter kernel and check for a local torch.py or torch directory."
        )
        raise RuntimeError(
            f"PyTorch and torchvision could not be imported together: {exc}.{recovery}"
        ) from exc
    return torch, torchvision


REPO_ROOT = find_mlx_checkout()
if REPO_ROOT is None:
    clone_root = Path("/content") if IN_COLAB else Path.cwd().resolve()
    clone_target = clone_root / "mlx"
    if clone_target.exists():
        raise RuntimeError(
            f"Cannot clone MLX because {clone_target} already exists but is not an MLX checkout. "
            "Move it, or start the notebook from an existing MLX checkout."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_target)])
    REPO_ROOT = clone_target.resolve()

repo_text = str(REPO_ROOT)
if repo_text not in sys.path:
    sys.path.insert(0, repo_text)

# Import Colab's bundled PyTorch before pip resolves the remaining dependencies.
if IN_COLAB:
    torch_runtime, torchvision_runtime = import_pytorch_runtime()
else:
    torch_runtime = torchvision_runtime = None

required_packages = [
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("cv2", "opencv-python"),
    ("numpy", "numpy"),
    ("rich", "rich>=13.7.0"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
]
missing_specs = [
    package_spec
    for module_name, package_spec in required_packages
    if importlib.util.find_spec(module_name) is None
]
if missing_specs:
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError(
            "Missing notebook dependencies: "
            + ", ".join(missing_specs)
            + ". Install them or set INSTALL_MISSING_DEPENDENCIES=True."
        )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *missing_specs]
    )
    importlib.invalidate_caches()

if torch_runtime is None or torchvision_runtime is None:
    torch_runtime, torchvision_runtime = import_pytorch_runtime()

from mlx.core.exceptions import MLXUserError
from mlx.modes.segmentation.models import supported_model_names

if MODEL not in supported_model_names():
    raise MLXUserError(
        f"Unsupported segmentation MODEL {MODEL!r}. Available models: "
        + ", ".join(supported_model_names())
    )

print(f"Runtime: {'Google Colab' if IN_COLAB else 'Jupyter'}")
print(f"MLX checkout: {REPO_ROOT}")
print(f"Model: {MODEL}")
print(
    f"PyTorch: {torch_runtime.__version__} ({torch_runtime.__file__}) | "
    f"torchvision: {torchvision_runtime.__version__}"
)

## 3. Select checkpoint storage

Google Drive mounting is limited to Colab. Local downloads and extracted data live under `<workspace-root>/<MODEL>`. Checkpoints live under that model workspace locally or under `<drive-checkpoint-root>/<MODEL>` in Colab.

In [ ]:
storage_mode = CHECKPOINT_STORAGE.strip().lower()
if storage_mode not in {"local", "google_drive"}:
    raise MLXUserError(
        "CHECKPOINT_STORAGE must be 'local' or 'google_drive'. "
        f"Received: {CHECKPOINT_STORAGE!r}"
    )

if LOCAL_WORKSPACE_ROOT:
    notebook_root = Path(LOCAL_WORKSPACE_ROOT).expanduser().resolve()
else:
    notebook_root = REPO_ROOT / "tmp" / "notebooks" / "train_semantic_segementation"
WORKSPACE_ROOT = notebook_root / MODEL
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

if storage_mode == "google_drive":
    if not IN_COLAB:
        raise MLXUserError(
            "Google Drive mounting is only available in Colab. For local Jupyter, "
            "set CHECKPOINT_STORAGE='local' and point LOCAL_WORKSPACE_ROOT at a synced directory."
        )
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
    checkpoint_root = Path(GOOGLE_DRIVE_CHECKPOINT_ROOT).expanduser()
    if not checkpoint_root.is_absolute():
        raise MLXUserError(
            "GOOGLE_DRIVE_CHECKPOINT_ROOT must be an absolute path beneath the mounted Drive."
        )
    OUTPUT_DIR = checkpoint_root / MODEL
else:
    OUTPUT_DIR = WORKSPACE_ROOT / "checkpoints"

OUTPUT_DIR = OUTPUT_DIR.resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model workspace: {WORKSPACE_ROOT}")
print(f"Model checkpoint directory: {OUTPUT_DIR}")

## 4. Download, extract, and validate the dataset

The URL is hashed into the archive and extraction directory names. Rerunning this cell reuses a completed download and valid extraction. ZIP members are checked before extraction to prevent writes outside the model workspace.

In [ ]:
import hashlib
import shutil
import zipfile
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

from mlx.modes.segmentation.data import IMAGE_EXTENSIONS


class PrepareSemanticSegmentationDataset:
    def __init__(self, *, archive_url: str, workspace: Path) -> None:
        self.archive_url = archive_url.strip()
        self.workspace = workspace

    def execute(self) -> Path:
        self._validate_url()
        source_id = hashlib.sha256(self.archive_url.encode("utf-8")).hexdigest()[:12]
        archive_path = self.workspace / "downloads" / f"dataset-{source_id}.zip"
        extraction_root = self.workspace / "datasets" / f"dataset-{source_id}"
        self._download(archive_path)
        self._extract(archive_path, extraction_root)
        return self._find_and_validate_dataset(extraction_root)

    def _validate_url(self) -> None:
        if not self.archive_url:
            raise MLXUserError(
                "DATASET_ZIP_URL is empty. Set it to a directly downloadable HTTP(S) ZIP file."
            )
        parsed = urlparse(self.archive_url)
        if parsed.scheme not in {"http", "https"} or not parsed.netloc:
            raise MLXUserError(
                "DATASET_ZIP_URL must be a directly downloadable HTTP(S) URL. "
                f"Received: {self.archive_url!r}"
            )

    def _download(self, archive_path: Path) -> None:
        if archive_path.is_file():
            print(f"Reusing downloaded archive: {archive_path}")
            return
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        partial_path = archive_path.with_suffix(".zip.part")
        request = Request(self.archive_url, headers={"User-Agent": "MLX-notebook"})
        print(f"Downloading dataset from {self.archive_url}")
        try:
            with urlopen(request, timeout=60) as response, partial_path.open("wb") as output:
                shutil.copyfileobj(response, output)
            partial_path.replace(archive_path)
        except (HTTPError, URLError, TimeoutError, OSError) as exc:
            partial_path.unlink(missing_ok=True)
            raise MLXUserError(
                f"Could not download the dataset ZIP from {self.archive_url}: {exc}. "
                "Check that the URL is public and points directly to a ZIP file."
            ) from exc

    def _extract(self, archive_path: Path, extraction_root: Path) -> None:
        if extraction_root.is_dir() and self._candidate_roots(extraction_root):
            print(f"Reusing extracted dataset: {extraction_root}")
            return
        extraction_root.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(archive_path) as archive:
                root = extraction_root.resolve()
                for member in archive.infolist():
                    target = (root / member.filename).resolve()
                    if target != root and root not in target.parents:
                        raise MLXUserError(
                            f"Unsafe path in dataset ZIP: {member.filename!r}. "
                            "Create an archive without absolute paths or '..' traversal."
                        )
                archive.extractall(root)
        except zipfile.BadZipFile as exc:
            raise MLXUserError(
                f"Downloaded file is not a valid ZIP archive: {archive_path}. "
                "Check that DATASET_ZIP_URL points directly to the archive."
            ) from exc

    def _candidate_roots(self, extraction_root: Path) -> list[Path]:
        return sorted(
            {
                images_dir.parent.parent.resolve()
                for images_dir in extraction_root.rglob("train/images")
                if images_dir.is_dir()
                and "__MACOSX" not in images_dir.parts
                and (images_dir.parent / "masks").is_dir()
                and (images_dir.parent.parent / "val" / "images").is_dir()
                and (images_dir.parent.parent / "val" / "masks").is_dir()
            }
        )

    def _find_and_validate_dataset(self, extraction_root: Path) -> Path:
        candidates = self._candidate_roots(extraction_root)
        if not candidates:
            raise MLXUserError(
                f"No paired segmentation dataset was found under {extraction_root}. "
                "Expected train/images, train/masks, val/images, and val/masks."
            )
        if len(candidates) > 1:
            found = ", ".join(str(path.relative_to(extraction_root)) for path in candidates)
            raise MLXUserError(
                "The ZIP contains multiple segmentation datasets. Package one dataset per archive. "
                f"Found: {found}"
            )
        dataset_root = candidates[0]
        for split in ("train", "val"):
            self._validate_pairs(dataset_root, split)
        test_dir = dataset_root / "test"
        if test_dir.exists():
            self._validate_pairs(dataset_root, "test")
        print(f"Validated segmentation dataset: {dataset_root}")
        return dataset_root

    def _validate_pairs(self, dataset_root: Path, split: str) -> None:
        images_dir = dataset_root / split / "images"
        masks_dir = dataset_root / split / "masks"
        if not images_dir.is_dir() or not masks_dir.is_dir():
            raise MLXUserError(
                f"Split '{split}' must contain images/ and masks/ directories under {dataset_root}."
            )
        image_stems = {
            path.stem for path in images_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        }
        mask_stems = {
            path.stem for path in masks_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        }
        if not image_stems:
            raise MLXUserError(f"No supported images were found in {images_dir}.")
        missing_masks = sorted(image_stems - mask_stems)
        missing_images = sorted(mask_stems - image_stems)
        if missing_masks or missing_images:
            details = []
            if missing_masks:
                details.append(f"missing masks for: {', '.join(missing_masks[:5])}")
            if missing_images:
                details.append(f"missing images for: {', '.join(missing_images[:5])}")
            raise MLXUserError(f"Image/mask mismatch in split '{split}': {'; '.join(details)}")

In [ ]:
DATASET_ROOT = PrepareSemanticSegmentationDataset(
    archive_url=DATASET_ZIP_URL,
    workspace=WORKSPACE_ROOT,
).execute()
print(f"Dataset root passed to MLX: {DATASET_ROOT}")

## 5. Resolve resume behavior

MLX resumes semantic-segmentation training from the model-specific `{MODEL}.last.pth` checkpoint. With `RESUME_LATEST=False`, this notebook refuses to reuse a model directory that already contains checkpoints.

In [ ]:
BEST_LOSS_CHECKPOINT = OUTPUT_DIR / f"{MODEL}.pth"
BEST_DICE_CHECKPOINT = OUTPUT_DIR / f"{MODEL}.best-dice.pth"
LAST_CHECKPOINT = OUTPUT_DIR / f"{MODEL}.last.pth"
existing_checkpoints = sorted(OUTPUT_DIR.glob("*.pth"))

if RESUME_LATEST and LAST_CHECKPOINT.is_file():
    RESUME_CHECKPOINT = LAST_CHECKPOINT
    print(f"MLX will resume {MODEL} from: {RESUME_CHECKPOINT}")
elif RESUME_LATEST:
    RESUME_CHECKPOINT = None
    print(f"Resume is enabled, but {LAST_CHECKPOINT.name} does not exist yet. Starting a new run.")
elif existing_checkpoints:
    found = ", ".join(path.name for path in existing_checkpoints)
    raise MLXUserError(
        f"RESUME_LATEST is disabled, but the {MODEL!r} checkpoint directory already contains: {found}. "
        "Move those checkpoints or select a different MODEL before starting a fresh run."
    )
else:
    RESUME_CHECKPOINT = None
    print("Resume is disabled and this model has no checkpoints. Starting a new run.")

## 6. Train

This cell uses MLX's `TrainSegmentationModel` command. For a resumed run, `EPOCHS` is the target total epoch count. Binary masks treat zero as background and any nonzero value as foreground; multiclass masks must contain class IDs from `0` through `NUM_CLASSES - 1`.

In [ ]:
import torch

from mlx.core.random import apply_global_seed
from mlx.modes.segmentation.requests import SegmentationRequest
from mlx.modes.segmentation.train import TrainSegmentationModel

if DEVICE == "auto":
    resolved_device = "cuda:0" if torch.cuda.is_available() else "cpu"
else:
    resolved_device = DEVICE

apply_global_seed(RANDOM_SEED)
request = SegmentationRequest(
    action="train",
    model=MODEL,
    model_path=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None,
    dataset_path=str(DATASET_ROOT),
    output_path=str(OUTPUT_DIR),
    device=resolved_device,
    width=IMAGE_WIDTH,
    height=IMAGE_HEIGHT,
    input_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    colored=COLORED,
    pretrained=PRETRAINED,
    num_classes=NUM_CLASSES,
    class_names=CLASS_NAMES,
    random_seed=RANDOM_SEED,
)

print(f"Training {MODEL} on: {resolved_device}", flush=True)
TrainSegmentationModel(request).execute()
print(f"Training command completed for {MODEL}.", flush=True)

## 7. Locate saved checkpoints and artifacts

The best validation-loss checkpoint is the normal inference checkpoint. The best-Dice checkpoint optimizes non-background overlap, and the last checkpoint contains resumable training state.

In [ ]:
print(f"Model: {MODEL}")
print(f"Artifact directory: {OUTPUT_DIR}")
print(f"Best validation-loss checkpoint: {BEST_LOSS_CHECKPOINT if BEST_LOSS_CHECKPOINT.is_file() else 'not found'}")
print(f"Best foreground-Dice checkpoint: {BEST_DICE_CHECKPOINT if BEST_DICE_CHECKPOINT.is_file() else 'not found'}")
print(f"Last resumable checkpoint: {LAST_CHECKPOINT if LAST_CHECKPOINT.is_file() else 'not found'}")
print(f"Training metrics: {OUTPUT_DIR / 'training.csv'}")
print(f"Training curves: {OUTPUT_DIR / 'training_curves.png'}")
print(f"Effective configuration: {OUTPUT_DIR / 'training_config.json'}")